In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import gc

<h2>Early Concatting, Transforming, and Merging</h2>

<h3>Concat Raw Datasets</h3>

In [31]:
# Concat all facility csv files into a single df

facility_folder = Path('Facility')

df_list = []
for file in facility_folder.iterdir():
    if file.is_file() and file.name.endswith('.csv'):
        df = pd.read_csv(file)
        df_list.append(df)

campd_facility = pd.concat(df_list, ignore_index=True)

output_path = Path('Facility/Concatted File/facility_2019_2024.csv')
campd_facility.to_csv(output_path, index=False)

In [87]:
# Concat all ND csv files into a single df

nd_folder = Path('ND/Hourly')

df_list = []
for file in nd_folder.iterdir():
    if file.is_file() and file.name.endswith('.csv'):
        df = pd.read_csv(file)
        df_list.append(df)

campd_nd = pd.concat(df_list, ignore_index=True)

# save full operating and non-operating dfs as one

campd_nd.to_csv('ND/Hourly/Concatted File/nd_2019_2024.csv', index=False)

del campd_nd
gc.collect()

0

In [22]:
# get all EIA data

eia_folder = Path('EIA')

df_list = []
for file in eia_folder.iterdir():
    if file.is_file() and file.name.endswith('.xlsx'):
        # extract the year from filename and add as column
        match = re.search(r'_Y(\d{4})\.xlsx', str(file))
        if match:
            year = int(match.group(1))
            df = pd.read_excel(file, sheet_name="Operable", header=1)
            df["Year"] = year
            df_list.append(df)

eia_generator = pd.concat(df_list, ignore_index=True)

# filter for rows in campd_ny, campd_wy, and campd_tx
eia_generator = eia_generator[eia_generator["Plant Code"].isin(campd_nd["Facility ID"])]

output_path = Path('EIA/Concatted File/eia_generator_2019_2023_ND.csv')
eia_generator.to_csv(output_path, index=False)

<h3>Transform Emission Data</h3>

In [3]:
# define data type dictionary for campd dataframes
campd_type_dict = {
    "State": "string",
    "Facility Name": "string",
    "Facility ID": "Int64",
    "Unit ID": "string",
    "Associated Stacks": "string",
    "Gross Load (MW)": "float64",
    "Steam Load (1000 lb/hr)": "float64",
    "SO2 Mass (lbs)": "float64",
    "SO2 Mass Measure Indicator": "string",
    "SO2 Rate (lbs/mmBtu)": "float64",
    "SO2 Rate Measure Indicator": "string",
    "CO2 Mass (short tons)": "float64",
    "CO2 Mass Measure Indicator": "string",
    "CO2 Rate (short tons/mmBtu)": "float64",
    "CO2 Rate Measure Indicator": "string",
    "NOx Mass (lbs)": "float64",
    "NOx Mass Measure Indicator": "string",
    "NOx Rate (lbs/mmBtu)": "float64",
    "NOx Rate Measure Indicator": "string",
    "Heat Input (mmBtu)": "float64",
    "Heat Input Measure Indicator": "string",
    "Primary Fuel Type": "string",
    "Secondary Fuel Type": "string",
    "Unit Type": "string",
    "SO2 Controls": "string",
    "NOx Controls": "string",
    "PM Controls": "string",
    "Hg Controls": "string",
    "Program Code": "string",
    "Hour": "Int64",
    "Operating Time": "float64"
}

<h4>North Dakota</h4>

In [89]:
campd_nd = pd.read_csv('ND/Hourly/Concatted File/nd_2019_2024.csv', dtype=campd_type_dict, parse_dates=["Date"])

In [90]:
campd_nd.shape

(1063128, 32)

<h3>Transform Facility Data</h3>

In [91]:
# read facility data and assign data types

campd_facility = pd.read_csv('Facility/Concatted File/facility_2019_2024.csv')

# limit to columns we need
campd_facility = campd_facility[["Year", "State", 'Facility ID', 'Unit ID', "EPA Region", 'Source Category', 'Latitude',
       'Longitude', 'SO2 Phase', 'NOx Phase',
       'Commercial Operation Date', 'Operating Status',
       'Max Hourly HI Rate (mmBtu/hr)',
       'Associated Generators & Nameplate Capacity (MWe)']]
campd_facility = campd_facility.astype({
    'Year': 'Int64',
    'Facility ID': 'Int64',
    'Unit ID': 'string',
    "State": "string",
    'EPA Region': 'Int64',
    "Source Category": 'string',
    'Latitude': 'float64',
    'Longitude': 'float64',
    'SO2 Phase': 'string',
    'NOx Phase': 'string',
    'Commercial Operation Date': 'string',
    'Operating Status': 'string',
    'Max Hourly HI Rate (mmBtu/hr)': 'float64',
    'Associated Generators & Nameplate Capacity (MWe)': 'string'
    })

# filter out rows that are not in campd_nd
campd_facility = campd_facility[
    campd_facility["Facility ID"].isin(campd_nd["Facility ID"])
]

In [92]:
# function to extract and sum nameplate capacities in facility data
def extract_and_sum_nameplate_capacity(data):
    data = str(data).strip()  # Convert data to string and remove leading/trailing whitespace
    matches = re.findall(r'\((\d+(?:\.\d+)?)\)', data) # find floats or integers in parentheses (can be float like 191.8 or integer like 170)

    if matches:
        total_sum = 0.0
        for match in matches:
            total_sum += float(match)
        #print("Total sum: ", total_sum)
        return total_sum
    else:
        #print("No matches found")
        return None

campd_facility['Nameplate Capacity (MWe)'] = campd_facility['Associated Generators & Nameplate Capacity (MWe)'].apply(extract_and_sum_nameplate_capacity)
campd_facility.to_csv('Facility/Concatted File/facility_2019_2024_ND.csv', index=False)

<h3>Merge Facility and Emission dfs</h3>

(and do early data cleaning based on this merge)

<h4>North Dakota</h4>

In [93]:
# Look for mismatched Unit IDs between campd_facility and campd_nd

# filter and get unique Facility-Unit pairs
fac_units_1 = campd_facility[
    campd_facility["State"] == "ND"
][["Facility ID", "Unit ID"]].drop_duplicates()

fac_units_2 = campd_nd[["Facility ID", "Unit ID"]].drop_duplicates()

# outer merge with indicator
merged = fac_units_1.merge(fac_units_2, on=["Facility ID", "Unit ID"], how="outer", indicator=True)

# filter mismatches
mismatches = merged[merged["_merge"] != "both"]

# count mismatched units per facility
mismatch_counts = mismatches.groupby("Facility ID").size().reset_index(name="Mismatched Unit Count")

# include the mismatching unit IDs as list
mismatch_units = (
    mismatches.groupby("Facility ID")["Unit ID"]
    .apply(list)
    .reset_index(name="Mismatching Unit IDs")
)

# merge count and unit list
result = mismatch_counts.merge(mismatch_units, on="Facility ID")

display(result)


,Facility ID,Mismatched Unit Count,Mismatching Unit IDs
0,2790,1,[B1]


In [94]:
# replace all unit IDs that are "001" or "01" with "1" to eliminate mismatches (see above)
campd_nd["Unit ID"] = campd_nd["Unit ID"].replace("001", "1")
campd_nd["Unit ID"] = campd_nd["Unit ID"].replace("01", "1")
campd_facility["Unit ID"] = campd_facility["Unit ID"].replace("001", "1")
campd_facility["Unit ID"] = campd_facility["Unit ID"].replace("01", "1")

In [95]:
# check after transformation

# filter and get unique Facility-Unit pairs
fac_units_1 = campd_facility[
    campd_facility["State"] == "ND"
][["Facility ID", "Unit ID"]].drop_duplicates()

fac_units_2 = campd_nd[["Facility ID", "Unit ID"]].drop_duplicates()

# outer merge with indicator
merged = fac_units_1.merge(fac_units_2, on=["Facility ID", "Unit ID"], how="outer", indicator=True)

# filter mismatches
mismatches = merged[merged["_merge"] != "both"]

# count mismatched units per facility
mismatch_counts = mismatches.groupby("Facility ID").size().reset_index(name="Mismatched Unit Count")

# include the mismatching unit IDs as list
mismatch_units = (
    mismatches.groupby("Facility ID")["Unit ID"]
    .apply(list)
    .reset_index(name="Mismatching Unit IDs")
)

# merge count and unit list
result = mismatch_counts.merge(mismatch_units, on="Facility ID")

display(result)

# the 001 -> 1 transformation fixed none here

,Facility ID,Mismatched Unit Count,Mismatching Unit IDs
0,2790,1,[B1]


In [96]:
campd_facility_nd = campd_facility[campd_facility["State"] == "ND"]
campd_facility_nd = campd_facility_nd[["Year", 'Facility ID', 'Unit ID', "EPA Region", 'Source Category', 'Latitude',
       'Longitude', 'SO2 Phase', 'NOx Phase',
       'Commercial Operation Date', 'Operating Status',
       'Max Hourly HI Rate (mmBtu/hr)',
       'Associated Generators & Nameplate Capacity (MWe)', 'Nameplate Capacity (MWe)']]
# add the year column to campd_ny for the join
campd_nd['Year'] = campd_nd['Date'].dt.year
campd_nd_facilities = pd.merge(campd_nd, campd_facility_nd, on=['Facility ID', 'Unit ID', 'Year'], how='left')

In [97]:
check = campd_nd_facilities.isna().sum()
check[check > 0]

Associated Stacks               1063128
Operating Time                     1021
Gross Load (MW)                  402284
Steam Load (1000 lb/hr)         1013673
SO2 Mass (lbs)                   352829
SO2 Mass Measure Indicator       352829
SO2 Rate (lbs/mmBtu)             352829
SO2 Rate Measure Indicator       352829
CO2 Mass (short tons)            352829
CO2 Mass Measure Indicator       352829
CO2 Rate (short tons/mmBtu)      352829
CO2 Rate Measure Indicator       352829
NOx Mass (lbs)                   352829
NOx Mass Measure Indicator       352829
NOx Rate (lbs/mmBtu)             352829
NOx Rate Measure Indicator       352829
Heat Input (mmBtu)               352829
Heat Input Measure Indicator     352829
Secondary Fuel Type              957912
SO2 Controls                     508584
NOx Controls                      28464
PM Controls                      508584
Hg Controls                      642264
NOx Phase                        589656
dtype: int64

In [98]:
campd_nd_facilities.to_csv('ND/Hourly/Concatted File/nd_2019_2024_with_facility.csv', index=False)

<h3>Transform EIA-860 Data</h3>

(has additional data about generator nameplate capacities and age, but needs to be aggregated to a unit level)

In [99]:
# read eia generator data
eia_generator = pd.read_csv('EIA/Concatted File/eia_generator_2019_2023_ND.csv')

# these cols have blank strings (" ") that should be nan
columns_to_clean = [
    "Nameplate Power Factor", "Summer Capacity (MW)", "Winter Capacity (MW)",
    "Minimum Load (MW)", "Planned Retirement Year"
]

# replace blank strings with nan (any strings with only whitespace)
eia_generator[columns_to_clean] = eia_generator[columns_to_clean].replace(r'^\s*$', np.nan, regex=True)

# now define full dtype dictionary
eia_generator_type_dict = {
    "Year": "Int64", 
    "Utility ID": "string",
    "Utility Name": "string",
    "Plant Code": "Int64",  
    "Plant Name": "string",
    "State": "string",
    "Generator ID": "string",
    "Technology": "string",
    "Prime Mover": "string",
    "Nameplate Capacity (MW)": "float64",
    "Nameplate Power Factor": "float64", 
    "Summer Capacity (MW)": "float64",  
    "Winter Capacity (MW)": "float64",
    "Minimum Load (MW)": "float64",
    "Operating Year": "Int64",
    "Planned Retirement Year": "float64"
}

# filter needed cols
eia_generator = eia_generator[["Year","Utility ID","Utility Name", "Plant Code", "Plant Name", "State", "Generator ID", "Technology", "Prime Mover", "Nameplate Capacity (MW)", "Nameplate Power Factor", "Summer Capacity (MW)", "Winter Capacity (MW)", "Minimum Load (MW)", "Operating Year", "Planned Retirement Year"]]

# cast all dtypes according to the dictionary
eia_generator = eia_generator.astype(eia_generator_type_dict)


<h4>old manual version without crosswalk</h4>

In [100]:
# Transform and aggregate to a unit-level (using facility data to add unit IDs)

# extract Generator IDs from the associated string (in campd_facility) and put them in a list 
def extract_generator_ids(data):
    data = str(data).strip()
    # format should be e. g. "G1 (170), G2 (191.8)"
    matches = re.findall(r'([A-Za-z0-9]+)\s*\(\d+(?:\.\d+)?\)', data)
    return matches if matches else []

campd_facility["Generator ID List"] = campd_facility["Associated Generators & Nameplate Capacity (MWe)"].apply(extract_generator_ids)

# explode so each row is Unit ID + one Generator ID -> create a year, facility, unit, generator mapping (a unit may have multiple generators)
unit_generator_map = campd_facility[["Year", "Facility ID", "Unit ID", "Generator ID List",]].explode("Generator ID List")
unit_generator_map = unit_generator_map.rename(columns={"Generator ID List": "Generator ID"})
display(unit_generator_map)

# merge with eia generator-level dataset
# (first rename Plant Code in eia_generator to Facility ID to match unit_generator_map)
eia_generator_renamed = eia_generator.rename(columns={"Plant Code": "Facility ID"})
# merge on Facility ID and Generator ID
joined = unit_generator_map.merge(eia_generator_renamed, on=["Year","Facility ID", "Generator ID"], how="inner")
display(joined)

,Year,Facility ID,Unit ID,Generator ID
1141,2019,2790,B1,1
1142,2019,2790,B2,2
1143,2019,2790,CT6,3
1144,2019,2817,1,1
1145,2019,2817,2,2
...,...,...,...,...
25190,2024,57943,CT2,02
25191,2024,57943,CT3,03
25192,2024,57943,CT4,04
25193,2024,57943,CT5,05


,Year,Facility ID,Unit ID,Generator ID,Utility ID,Utility Name,Plant Name,State,Technology,Prime Mover,Nameplate Capacity (MW),Nameplate Power Factor,Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Operating Year,Planned Retirement Year
0,2019,2790,B1,1,12199,Montana-Dakota Utilities Co,R M Heskett,ND,Conventional Steam Coal,ST,40.0,1.00,29.5,29.5,6.5,1954,2022.0
1,2019,2790,B2,2,12199,Montana-Dakota Utilities Co,R M Heskett,ND,Conventional Steam Coal,ST,75.0,0.85,74.8,74.8,29.3,1963,2022.0
2,2019,2790,CT6,3,12199,Montana-Dakota Utilities Co,R M Heskett,ND,Natural Gas Fired Combustion Turbine,GT,88.0,0.85,88.0,101.0,8.0,2014,NaN
3,2019,2817,1,1,1307,Basin Electric Power Coop,Leland Olds,ND,Conventional Steam Coal,ST,216.0,0.85,221.0,221.0,110.0,1966,NaN
4,2019,2817,2,2,1307,Basin Electric Power Coop,Leland Olds,ND,Conventional Steam Coal,ST,440.0,0.90,445.0,445.0,220.0,1975,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,2023,6030,2,2,65146,Rainbow Energy Center,Coal Creek,ND,Conventional Steam Coal,ST,604.8,0.90,573.9,573.9,285.0,1980,NaN
57,2023,6469,B1,1,1307,Basin Electric Power Coop,Antelope Valley,ND,Conventional Steam Coal,ST,477.0,0.90,450.0,450.0,180.0,1984,NaN
58,2023,6469,B2,2,1307,Basin Electric Power Coop,Antelope Valley,ND,Conventional Steam Coal,ST,477.0,0.90,450.0,450.0,180.0,1986,NaN
59,2023,8222,B1,1,14232,Otter Tail Power Co,Coyote,ND,Conventional Steam Coal,ST,450.0,0.90,427.0,427.0,150.0,1981,NaN


In [101]:
# aggregate the resulting generator-level df to a unit-level df

agg_funcs = {
    'Nameplate Capacity (MW)': 'sum',
    'Summer Capacity (MW)': 'sum',
    'Winter Capacity (MW)': 'sum',
    'Minimum Load (MW)': 'sum',
    'Operating Year': ['min', 'max', 'mean'], # get operating year of the oldest (min) and newest (max) generator in the unit as well as the average across all gens
    'Planned Retirement Year': 'max',
}

unit_level = joined.groupby(['Year', 'Facility ID', 'Unit ID']).agg(agg_funcs).reset_index()

display(unit_level)

# flatten multi-index columns (only keep the aggregation operator (min, max, mean) for Operating Year)
unit_level.columns = [
    f"{col[0]}_{col[1]}" if col[0] == "Operating Year" else col[0]
    for col in unit_level.columns.to_flat_index()
]

unit_level

Year Facility ID Unit ID Nameplate Capacity (MW) Summer Capacity (MW)  \
                                                 sum                  sum   
0   2019        2790      B1                    40.0                 29.5   
1   2019        2790      B2                    75.0                 74.8   
2   2019        2790     CT6                    88.0                 88.0   
3   2019        2817       1                   216.0                221.0   
4   2019        2817       2                   440.0                445.0   
..   ...         ...     ...                     ...                  ...   
56  2023        6030       2                   604.8                573.9   
57  2023        6469      B1                   477.0                450.0   
58  2023        6469      B2                   477.0                450.0   
59  2023        8222      B1                   450.0                427.0   
60  2023       56786       1                   106.2                 97.1   

   Winter Capacity (MW) Minimum Load (MW) Operating Year                \
                    sum               sum            min   max    mean   
0                  29.5               6.5           1954  1954  1954.0   
1                  74.8              29.3           1963  1963  1963.0   
2                 101.0               8.0           2014  2014  2014.0   
3                 221.0             110.0           1966  1966  1966.0   
4                 445.0             220.0           1975  1975  1975.0   
..                  ...               ...            ...   ...     ...   
56                573.9             285.0           1980  1980  1980.0   
57                450.0             180.0           1984  1984  1984.0   
58                450.0             180.0           1986  1986  1986.0   
59                427.0             150.0           1981  1981  1981.0   
60                 97.1              25.0           2014  2014  2014.0   

   Planned Retirement Year  
                       max  
0                   2022.0  
1                   2022.0  
2                      NaN  
3                      NaN  
4                      NaN  
..                     ...  
56                     NaN  
57                     NaN  
58                     NaN  
59                     NaN  
60                     NaN  

[61 rows x 11 columns]

,Year,Facility ID,Unit ID,Nameplate Capacity (MW),Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Operating Year_min,Operating Year_max,Operating Year_mean,Planned Retirement Year
0,2019,2790,B1,40.0,29.5,29.5,6.5,1954,1954,1954.0,2022.0
1,2019,2790,B2,75.0,74.8,74.8,29.3,1963,1963,1963.0,2022.0
2,2019,2790,CT6,88.0,88.0,101.0,8.0,2014,2014,2014.0,NaN
3,2019,2817,1,216.0,221.0,221.0,110.0,1966,1966,1966.0,NaN
4,2019,2817,2,440.0,445.0,445.0,220.0,1975,1975,1975.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...
56,2023,6030,2,604.8,573.9,573.9,285.0,1980,1980,1980.0,NaN
57,2023,6469,B1,477.0,450.0,450.0,180.0,1984,1984,1984.0,NaN
58,2023,6469,B2,477.0,450.0,450.0,180.0,1986,1986,1986.0,NaN
59,2023,8222,B1,450.0,427.0,427.0,150.0,1981,1981,1981.0,NaN


In [102]:
# set data types again for the final unit_level dataframe
unit_level = unit_level.astype({
    'Year': 'Int64',
    'Facility ID': 'Int64',
    'Unit ID': 'string',
    'Nameplate Capacity (MW)': 'float64',
    'Summer Capacity (MW)': 'float64',
    'Winter Capacity (MW)': 'float64',
    'Minimum Load (MW)': 'float64',
    'Operating Year_min': 'Int64',
    'Operating Year_max': 'Int64',
    'Operating Year_mean': 'Int64',
    'Planned Retirement Year': 'Int64'
})

In [103]:
# Artificially extend the unit_level dataframe to include 2024 data (just copy the 2023 data and set Year to 2024)
unit_level_2023 = unit_level[unit_level["Year"] == 2023].copy()
unit_level_2024 = unit_level_2023.copy()
unit_level_2024["Year"] = 2024
unit_level_extended = pd.concat([unit_level, unit_level_2024], ignore_index=True)

In [104]:
# Calc min, max, and mean ages of the generators in the unit
unit_level_extended['unit_age_min'] = unit_level_extended["Year"] - unit_level_extended['Operating Year_min']
unit_level_extended['unit_age_max'] = unit_level_extended["Year"] - unit_level_extended['Operating Year_max']
unit_level_extended['unit_age_mean'] = unit_level_extended["Year"] - unit_level_extended['Operating Year_mean']

In [105]:
# save to csv
unit_level_extended.to_csv('EIA/Concatted File/eia_unit_level_2019_2024_ND.csv', index=False)

In [106]:
unit_level_extended_manually = pd.read_csv('EIA/Concatted File/eia_unit_level_2019_2024_ND.csv')
unit_level_extended_manually

,Year,Facility ID,Unit ID,Nameplate Capacity (MW),Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Operating Year_min,Operating Year_max,Operating Year_mean,Planned Retirement Year,unit_age_min,unit_age_max,unit_age_mean
0,2019,2790,B1,40.0,29.5,29.5,6.5,1954,1954,1954,2022.0,65,65,65
1,2019,2790,B2,75.0,74.8,74.8,29.3,1963,1963,1963,2022.0,56,56,56
2,2019,2790,CT6,88.0,88.0,101.0,8.0,2014,2014,2014,NaN,5,5,5
3,2019,2817,1,216.0,221.0,221.0,110.0,1966,1966,1966,NaN,53,53,53
4,2019,2817,2,440.0,445.0,445.0,220.0,1975,1975,1975,NaN,44,44,44
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,2024,6030,2,604.8,573.9,573.9,285.0,1980,1980,1980,NaN,44,44,44
68,2024,6469,B1,477.0,450.0,450.0,180.0,1984,1984,1984,NaN,40,40,40
69,2024,6469,B2,477.0,450.0,450.0,180.0,1986,1986,1986,NaN,38,38,38
70,2024,8222,B1,450.0,427.0,427.0,150.0,1981,1981,1981,NaN,43,43,43


<h3>Merge Emission data with EIA data</h3>

In [107]:
campd_nd_facilities = pd.read_csv('ND/Hourly/Concatted File/nd_2019_2024_with_facility.csv', dtype=campd_type_dict, date_format=["Date"])

C:\Users\marce\AppData\Local\Temp\ipykernel_5092\119115805.py:1: DtypeWarning: Columns (38) have mixed types. Specify dtype option on import or set low_memory=False.
  campd_nd_facilities = pd.read_csv('ND/Hourly/Concatted File/nd_2019_2024_with_facility.csv', dtype=campd_type_dict, date_format=["Date"])


In [108]:
complete_campd_nd = pd.merge(campd_nd_facilities, unit_level_extended_manually, on=['Year', 'Facility ID', 'Unit ID'], how='left')

In [109]:
# Filter where the values are different (excluding NaNs where both are the same)
mismatch_df = complete_campd_nd[
    complete_campd_nd["Nameplate Capacity (MWe)"] != complete_campd_nd["Nameplate Capacity (MW)"]
][["Year", "Nameplate Capacity (MWe)", "Nameplate Capacity (MW)"]].drop_duplicates()

# Count where either column is NaN
nan_mwe = complete_campd_nd["Nameplate Capacity (MWe)"].isna().sum()
nan_mw = complete_campd_nd["Nameplate Capacity (MW)"].isna().sum()

# Display results
print("Unique mismatching rows (with year):")
display(mismatch_df)

print(f"\nNaN counts:")
print(f"Nameplate Capacity (MWe): {nan_mwe}")
print(f"Nameplate Capacity (MW): {nan_mw}")

Unique mismatching rows (with year):


,Year,Nameplate Capacity (MWe),Nameplate Capacity (MW)
35040,2019,270.0,257.0
43800,2019,518.0,477.0
52560,2019,605.0,604.8
70080,2019,435.0,477.0
96360,2019,99.0,106.2
105120,2019,45.2,NaN
210336,2020,270.0,257.0
219120,2020,518.0,477.0
227904,2020,605.0,604.8
245472,2020,435.0,477.0



NaN counts:
Nameplate Capacity (MWe): 0
Nameplate Capacity (MW): 458136


In [110]:
# Create a column for absolute difference
complete_campd_nd["Capacity Difference"] = (
    complete_campd_nd["Nameplate Capacity (MWe)"] - complete_campd_nd["Nameplate Capacity (MW)"]
).abs()

# Drop rows with NaNs in either of the capacity columns
diff_df = complete_campd_nd.dropna(subset=["Nameplate Capacity (MWe)", "Nameplate Capacity (MW)"])

# Calculate average difference
average_difference = diff_df["Capacity Difference"].mean()
print(f"Average Capacity Difference: {average_difference:.2f} MW")

# Sort by largest difference
top_differences = diff_df.sort_values(by="Capacity Difference", ascending=False)

# Display top 10 mismatches
display(top_differences[["Year", "Facility ID", "Unit ID", 
                         "Nameplate Capacity (MWe)", "Nameplate Capacity (MW)", 
                         "Capacity Difference"]].head(10))


Average Capacity Difference: 12.66 MW


,Year,Facility ID,Unit ID,Nameplate Capacity (MWe),Nameplate Capacity (MW),Capacity Difference
260741,2020,6469,B2,435.0,477.0,42.0
262993,2020,6469,B2,435.0,477.0,42.0
262986,2020,6469,B2,435.0,477.0,42.0
262987,2020,6469,B2,435.0,477.0,42.0
262988,2020,6469,B2,435.0,477.0,42.0
262989,2020,6469,B2,435.0,477.0,42.0
262990,2020,6469,B2,435.0,477.0,42.0
262991,2020,6469,B2,435.0,477.0,42.0
262992,2020,6469,B2,435.0,477.0,42.0
262994,2020,6469,B2,435.0,477.0,42.0


In [111]:
# fill EIA columns with na if it capacity difference exceeds a certain threshold

# rename the EIA column to make it clearer
for df in [complete_campd_nd]:
    df.rename(columns={
        "Nameplate Capacity (MW)": "Nameplate Capacity (MW) - EIA"
    }, inplace=True)

threshold = 30 
columns_to_null = [
    "Nameplate Capacity (MW) - EIA", 
    "Summer Capacity (MW)",
    "Winter Capacity (MW)",
    "Minimum Load (MW)"
]

# Recalculate capacity difference
complete_campd_nd["Capacity Difference"] = (
    complete_campd_nd["Nameplate Capacity (MWe)"] - complete_campd_nd["Nameplate Capacity (MW) - EIA"]
).abs()

# Identify rows with unreasonable differences
condition = complete_campd_nd["Capacity Difference"] > threshold

# Set selected columns to NaN where condition is met
complete_campd_nd.loc[condition, columns_to_null] = np.nan

In [112]:
navals = complete_campd_nd[["Nameplate Capacity (MW) - EIA", "Nameplate Capacity (MWe)"]].isna().sum()
navals_percent = (navals / len(complete_campd_nd)) * 100
navals_percent.sort_values(ascending=False)

Nameplate Capacity (MW) - EIA    57.938461
Nameplate Capacity (MWe)          0.000000
dtype: float64

In [113]:
complete_campd_nd.to_csv('ND/Hourly/Concatted File/nd_2019_2024_with_manualEIA.csv', index=False)

<h2>Data Cleaning of Final Datasets</h2>

In [114]:
# relevant columns
complete_campd_columns = [
    "State",
    "Facility Name",
    "Facility ID",
    "Unit ID",
    "Gross Load (MW)",
    "Steam Load (1000 lb/hr)",
    "SO2 Mass (lbs)",
    "SO2 Mass Measure Indicator",
    "SO2 Rate (lbs/mmBtu)",
    "SO2 Rate Measure Indicator",
    "CO2 Mass (short tons)",
    "CO2 Mass Measure Indicator",
    "CO2 Rate (short tons/mmBtu)",
    "CO2 Rate Measure Indicator",
    "NOx Mass (lbs)",
    "NOx Mass Measure Indicator",
    "NOx Rate (lbs/mmBtu)",
    "NOx Rate Measure Indicator",
    "Heat Input (mmBtu)",
    "Heat Input Measure Indicator",
    "Primary Fuel Type",
    "Secondary Fuel Type",
    "Unit Type",
    "SO2 Controls",
    "NOx Controls",
    "Program Code",
    "Hour",
    "Year",
    "Date",
    "Operating Time",
    "Source Category",
    "Latitude",
    "Longitude",
    "SO2 Phase",
    "NOx Phase",
    "Operating Status",
    "Max Hourly HI Rate (mmBtu/hr)",
    "Associated Generators & Nameplate Capacity (MWe)",
    "Nameplate Capacity (MWe)",
    "Nameplate Capacity (MW) - EIA",
    "Summer Capacity (MW)",
    "Winter Capacity (MW)",
    "Minimum Load (MW)",
    "Planned Retirement Year",
    "unit_age_min",
    "unit_age_max",
    "unit_age_mean"
]

# Filtered dtype dictionary
complete_campd_type_dict = {
    "State": "string",
    "Facility Name": "string",
    "Facility ID": "Int64",
    "Unit ID": "string",
    "Gross Load (MW)": "float64",
    "Steam Load (1000 lb/hr)": "float64",
    "SO2 Mass (lbs)": "float64",
    "SO2 Mass Measure Indicator": "string",
    "SO2 Rate (lbs/mmBtu)": "float64",
    "SO2 Rate Measure Indicator": "string",
    "CO2 Mass (short tons)": "float64",
    "CO2 Mass Measure Indicator": "string",
    "CO2 Rate (short tons/mmBtu)": "float64",
    "CO2 Rate Measure Indicator": "string",
    "NOx Mass (lbs)": "float64",
    "NOx Mass Measure Indicator": "string",
    "NOx Rate (lbs/mmBtu)": "float64",
    "NOx Rate Measure Indicator": "string",
    "Heat Input (mmBtu)": "float64",
    "Heat Input Measure Indicator": "string",
    "Primary Fuel Type": "string",
    "Secondary Fuel Type": "string",
    "Unit Type": "string",
    "SO2 Controls": "string",
    "NOx Controls": "string",
    "Program Code": "string",
    "Hour": "Int64",
    "Year": "Int64",
    "Operating Time": "float64",
    "Source Category": "string",
    "Latitude": "float64",
    "Longitude": "float64",
    "SO2 Phase": "string",
    "NOx Phase": "string",
    "Operating Status": "string",
    "Max Hourly HI Rate (mmBtu/hr)": "float64",
    "Associated Generators & Nameplate Capacity (MWe)": "string",
    "Nameplate Capacity (MW)": "float64",
    "Nameplate Capacity (MW) - EIA": "float64",
    "Summer Capacity (MW)": "float64",
    "Winter Capacity (MW)": "float64",
    "Minimum Load (MW)": "float64",
    "Planned Retirement Year": "Int64",
    "unit_age_min": "Int64",
    "unit_age_max": "Int64",
    "unit_age_mean": "float64"
}


<h3>North Dakota</h3>

In [115]:
complete_campd_nd = pd.read_csv(
    'ND/Hourly/Concatted File/nd_2019_2024_with_manualEIA.csv',  
    usecols=complete_campd_columns,
    dtype=complete_campd_type_dict,
    parse_dates=["Date"]
)

# extract day and month as new col from date
complete_campd_nd["Month"] = complete_campd_nd["Date"].dt.month
complete_campd_nd["Day"] = complete_campd_nd["Date"].dt.day

display(complete_campd_nd.head())
print(complete_campd_nd.dtypes)

,State,Facility Name,Facility ID,Unit ID,Date,Hour,Operating Time,Gross Load (MW),Steam Load (1000 lb/hr),SO2 Mass (lbs),...,Nameplate Capacity (MW) - EIA,Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Planned Retirement Year,unit_age_min,unit_age_max,unit_age_mean,Month,Day
0,ND,R M Heskett,2790,B2,2019-01-01,0,1.0,59.0,NaN,251.7,...,75.0,74.8,74.8,29.3,2022,56,56,56.0,1,1
1,ND,R M Heskett,2790,B2,2019-01-01,1,1.0,59.0,NaN,264.7,...,75.0,74.8,74.8,29.3,2022,56,56,56.0,1,1
2,ND,R M Heskett,2790,B2,2019-01-01,2,1.0,60.0,NaN,302.6,...,75.0,74.8,74.8,29.3,2022,56,56,56.0,1,1
3,ND,R M Heskett,2790,B2,2019-01-01,3,1.0,59.0,NaN,229.3,...,75.0,74.8,74.8,29.3,2022,56,56,56.0,1,1
4,ND,R M Heskett,2790,B2,2019-01-01,4,1.0,58.0,NaN,242.9,...,75.0,74.8,74.8,29.3,2022,56,56,56.0,1,1


State                                               string[python]
Facility Name                                       string[python]
Facility ID                                                  Int64
Unit ID                                             string[python]
Date                                                datetime64[ns]
Hour                                                         Int64
Operating Time                                             float64
Gross Load (MW)                                            float64
Steam Load (1000 lb/hr)                                    float64
SO2 Mass (lbs)                                             float64
SO2 Mass Measure Indicator                          string[python]
SO2 Rate (lbs/mmBtu)                                       float64
SO2 Rate Measure Indicator                          string[python]
CO2 Mass (short tons)                                      float64
CO2 Mass Measure Indicator                          string[pyt

In [116]:
print(complete_campd_nd.shape)
null_nd = complete_campd_nd.isnull().sum()
null_nd = null_nd[null_nd > 0].sort_values(ascending=False)
display(null_nd)

# Give null vals as percentage of total rows
total_rows = complete_campd_nd.shape[0]
null_percentages = (null_nd / total_rows) * 100

null_percentages.sort_values(ascending=False)

(1063128, 49)


Steam Load (1000 lb/hr)          1013673
Planned Retirement Year          1001736
Secondary Fuel Type               957912
Minimum Load (MW)                 615960
Winter Capacity (MW)              615960
Summer Capacity (MW)              615960
Nameplate Capacity (MW) - EIA     615960
NOx Phase                         589656
SO2 Controls                      508584
unit_age_mean                     458136
unit_age_max                      458136
unit_age_min                      458136
Gross Load (MW)                   402284
Heat Input (mmBtu)                352829
Heat Input Measure Indicator      352829
NOx Rate Measure Indicator        352829
NOx Rate (lbs/mmBtu)              352829
NOx Mass Measure Indicator        352829
NOx Mass (lbs)                    352829
CO2 Rate Measure Indicator        352829
CO2 Rate (short tons/mmBtu)       352829
CO2 Mass Measure Indicator        352829
CO2 Mass (short tons)             352829
SO2 Rate Measure Indicator        352829
SO2 Rate (lbs/mm

Steam Load (1000 lb/hr)          95.348161
Planned Retirement Year          94.225343
Secondary Fuel Type              90.103167
Minimum Load (MW)                57.938461
Winter Capacity (MW)             57.938461
Summer Capacity (MW)             57.938461
Nameplate Capacity (MW) - EIA    57.938461
NOx Phase                        55.464253
SO2 Controls                     47.838454
unit_age_min                     43.093212
unit_age_max                     43.093212
unit_age_mean                    43.093212
Gross Load (MW)                  37.839658
CO2 Rate (short tons/mmBtu)      33.187819
SO2 Mass (lbs)                   33.187819
SO2 Mass Measure Indicator       33.187819
SO2 Rate (lbs/mmBtu)             33.187819
SO2 Rate Measure Indicator       33.187819
CO2 Mass (short tons)            33.187819
CO2 Mass Measure Indicator       33.187819
Heat Input Measure Indicator     33.187819
CO2 Rate Measure Indicator       33.187819
NOx Mass (lbs)                   33.187819
NOx Mass Me

In [117]:
# handle categorical null values appropriately

cat_cols_with_na_none = [
    "SO2 Controls",
    "NOx Phase",
    "Secondary Fuel Type",
    "NOx Controls",
    "SO2 Phase"
]

cat_cols_with_na_unknown = [
    "Operating Status",
    "Associated Generators & Nameplate Capacity (MWe)",
    "Source Category",
    "CO2 Rate Measure Indicator",
    "SO2 Rate Measure Indicator",
    "NOx Rate Measure Indicator", "Program Code"
]

# here nan means that no data was likely purposely reported -> fill with "Nothing"
for col in cat_cols_with_na_none:
    complete_campd_nd[col] = complete_campd_nd[col].fillna("Nothing").astype("string")

# here nan means that no data was likely reported -> fill with "Missing"
for col in cat_cols_with_na_unknown:
    complete_campd_nd[col] = complete_campd_nd[col].fillna("Missing").astype("string")

num_cols_with_na = [
    "Planned Retirement Year",
    "unit_age_max",
    "unit_age_min",
    "unit_age_mean",
    "Minimum Load (MW)",
    "Winter Capacity (MW)",
    "Summer Capacity (MW)",
    #"Nameplate Capacity (MW) - EIA", # dropped later
    "Longitude",
    "Latitude",
    "Max Hourly HI Rate (mmBtu/hr)",
    "Nameplate Capacity (MWe)"
]

# we drop nameplate capacity (MW) - EIA (redundant with Nameplate Capacity (MWe) and has many nulls)
complete_campd_nd = complete_campd_nd.drop(columns=["Nameplate Capacity (MW) - EIA"])

print("Null value percentages in complete_campd_nd after adjustments:")
total_rows = complete_campd_nd.shape[0]
null_nd = complete_campd_nd.isnull().sum()
null_nd = null_nd[null_nd > 0].sort_values(ascending=False)
null_percentages = (null_nd / total_rows) * 100
null_percentages.sort_values(ascending=False)

Null value percentages in complete_campd_nd after adjustments:


Steam Load (1000 lb/hr)         95.348161
Planned Retirement Year         94.225343
Minimum Load (MW)               57.938461
Winter Capacity (MW)            57.938461
Summer Capacity (MW)            57.938461
unit_age_mean                   43.093212
unit_age_max                    43.093212
unit_age_min                    43.093212
Gross Load (MW)                 37.839658
CO2 Mass Measure Indicator      33.187819
SO2 Mass (lbs)                  33.187819
SO2 Mass Measure Indicator      33.187819
SO2 Rate (lbs/mmBtu)            33.187819
CO2 Mass (short tons)           33.187819
Heat Input Measure Indicator    33.187819
CO2 Rate (short tons/mmBtu)     33.187819
NOx Mass (lbs)                  33.187819
NOx Rate (lbs/mmBtu)            33.187819
NOx Mass Measure Indicator      33.187819
Heat Input (mmBtu)              33.187819
Operating Time                   0.096037
dtype: float64

In [118]:
# measurement methods
print("Measurement methods in complete_campd_nd:")
display(complete_campd_nd.groupby("CO2 Mass Measure Indicator")["CO2 Mass (short tons)"].count())
display(complete_campd_nd.groupby("NOx Mass Measure Indicator")["NOx Mass Measure Indicator"].count())
display(complete_campd_nd.groupby("SO2 Mass Measure Indicator")["SO2 Mass Measure Indicator"].count())

Measurement methods in complete_campd_nd:


CO2 Mass Measure Indicator
Measured                   707200
Measured and Substitute      2967
Substitute                    132
Name: CO2 Mass (short tons), dtype: int64

NOx Mass Measure Indicator
Calculated    710299
Name: NOx Mass Measure Indicator, dtype: int64

SO2 Mass Measure Indicator
Measured                   708539
Measured and Substitute      1597
Substitute                    163
Name: SO2 Mass Measure Indicator, dtype: int64

In [119]:
# calculate hourly capacity factor
complete_campd_nd["hourly_capacity_factor"] = (
    complete_campd_nd["Gross Load (MW)"] * complete_campd_nd["Operating Time"]
) / complete_campd_nd["Nameplate Capacity (MWe)"]

complete_campd_nd.head(5)

,State,Facility Name,Facility ID,Unit ID,Date,Hour,Operating Time,Gross Load (MW),Steam Load (1000 lb/hr),SO2 Mass (lbs),...,Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Planned Retirement Year,unit_age_min,unit_age_max,unit_age_mean,Month,Day,hourly_capacity_factor
0,ND,R M Heskett,2790,B2,2019-01-01,0,1.0,59.0,NaN,251.7,...,74.8,74.8,29.3,2022,56,56,56.0,1,1,0.786667
1,ND,R M Heskett,2790,B2,2019-01-01,1,1.0,59.0,NaN,264.7,...,74.8,74.8,29.3,2022,56,56,56.0,1,1,0.786667
2,ND,R M Heskett,2790,B2,2019-01-01,2,1.0,60.0,NaN,302.6,...,74.8,74.8,29.3,2022,56,56,56.0,1,1,0.800000
3,ND,R M Heskett,2790,B2,2019-01-01,3,1.0,59.0,NaN,229.3,...,74.8,74.8,29.3,2022,56,56,56.0,1,1,0.786667
4,ND,R M Heskett,2790,B2,2019-01-01,4,1.0,58.0,NaN,242.9,...,74.8,74.8,29.3,2022,56,56,56.0,1,1,0.773333


In [130]:
# calculate hourly ramping factor
all_hours_nd = complete_campd_nd
all_hours_nd["Gross Load (MW)"] = all_hours_nd["Gross Load (MW)"].fillna(0)


# Sort data by unit and time to calculate t-1 values
all_hours_nd = all_hours_nd.sort_values(["Facility ID", "Unit ID", "Date", "Hour"])

# Calculate lagged gross load per unit (get previous row's value for gross load)
all_hours_nd["gross_load_lag1"] = all_hours_nd.groupby(["Facility ID", "Unit ID"])["Gross Load (MW)"].shift(1)

# calc ramping factor
all_hours_nd["hourly_ramping_factor"] = (
    all_hours_nd["Gross Load (MW)"] - all_hours_nd["gross_load_lag1"]
) / all_hours_nd["Nameplate Capacity (MWe)"]

display(all_hours_nd.head(10))

complete_campd_nd = complete_campd_nd.merge(all_hours_nd[["Facility ID", "Unit ID", "Date", "Hour", "hourly_ramping_factor"]],
                        on=["Facility ID", "Unit ID", "Date", "Hour"],
                        how="left")
complete_campd_nd.head(5)

,State,Facility Name,Facility ID,Unit ID,Date,Hour,Operating Time,Gross Load (MW),Steam Load (1000 lb/hr),SO2 Mass (lbs),...,Minimum Load (MW),Planned Retirement Year,unit_age_min,unit_age_max,unit_age_mean,Month,Day,hourly_capacity_factor,gross_load_lag1,hourly_ramping_factor
880848,ND,R M Heskett,2790,12,2024-04-01,0,NaN,0.0,NaN,NaN,...,NaN,<NA>,<NA>,<NA>,NaN,4,1,NaN,NaN,NaN
880849,ND,R M Heskett,2790,12,2024-04-01,1,NaN,0.0,NaN,NaN,...,NaN,<NA>,<NA>,<NA>,NaN,4,1,NaN,0.0,0.0
880850,ND,R M Heskett,2790,12,2024-04-01,2,NaN,0.0,NaN,NaN,...,NaN,<NA>,<NA>,<NA>,NaN,4,1,NaN,0.0,0.0
880851,ND,R M Heskett,2790,12,2024-04-01,3,NaN,0.0,NaN,NaN,...,NaN,<NA>,<NA>,<NA>,NaN,4,1,NaN,0.0,0.0
880852,ND,R M Heskett,2790,12,2024-04-01,4,NaN,0.0,NaN,NaN,...,NaN,<NA>,<NA>,<NA>,NaN,4,1,NaN,0.0,0.0
880853,ND,R M Heskett,2790,12,2024-04-01,5,NaN,0.0,NaN,NaN,...,NaN,<NA>,<NA>,<NA>,NaN,4,1,NaN,0.0,0.0
880854,ND,R M Heskett,2790,12,2024-04-01,6,NaN,0.0,NaN,NaN,...,NaN,<NA>,<NA>,<NA>,NaN,4,1,NaN,0.0,0.0
880855,ND,R M Heskett,2790,12,2024-04-01,7,NaN,0.0,NaN,NaN,...,NaN,<NA>,<NA>,<NA>,NaN,4,1,NaN,0.0,0.0
880856,ND,R M Heskett,2790,12,2024-04-01,8,NaN,0.0,NaN,NaN,...,NaN,<NA>,<NA>,<NA>,NaN,4,1,NaN,0.0,0.0
880857,ND,R M Heskett,2790,12,2024-04-01,9,NaN,0.0,NaN,NaN,...,NaN,<NA>,<NA>,<NA>,NaN,4,1,NaN,0.0,0.0


,State,Facility Name,Facility ID,Unit ID,Date,Hour,Operating Time,Gross Load (MW),Steam Load (1000 lb/hr),SO2 Mass (lbs),...,Winter Capacity (MW),Minimum Load (MW),Planned Retirement Year,unit_age_min,unit_age_max,unit_age_mean,Month,Day,hourly_capacity_factor,hourly_ramping_factor
0,ND,R M Heskett,2790,B2,2019-01-01,0,1.0,59.0,NaN,251.7,...,74.8,29.3,2022,56,56,56.0,1,1,0.786667,NaN
1,ND,R M Heskett,2790,B2,2019-01-01,1,1.0,59.0,NaN,264.7,...,74.8,29.3,2022,56,56,56.0,1,1,0.786667,0.000000
2,ND,R M Heskett,2790,B2,2019-01-01,2,1.0,60.0,NaN,302.6,...,74.8,29.3,2022,56,56,56.0,1,1,0.800000,0.013333
3,ND,R M Heskett,2790,B2,2019-01-01,3,1.0,59.0,NaN,229.3,...,74.8,29.3,2022,56,56,56.0,1,1,0.786667,-0.013333
4,ND,R M Heskett,2790,B2,2019-01-01,4,1.0,58.0,NaN,242.9,...,74.8,29.3,2022,56,56,56.0,1,1,0.773333,-0.013333


In [131]:
complete_campd_nd.to_csv('ND/Hourly/Concatted File/nd_2019_2024_complete.csv', index=False)

<h2>Transform and merge EIA-860 Data using crosswalk</h2>

In [132]:
# read eia generator data
eia_generator = pd.read_csv('EIA/Concatted File/eia_generator_2019_2023_ND.csv')

# these cols have blank strings (" ") that should be nan
columns_to_clean = [
    "Nameplate Power Factor", "Summer Capacity (MW)", "Winter Capacity (MW)",
    "Minimum Load (MW)", "Planned Retirement Year"
]

# replace blank strings with nan (any strings with only whitespace)
eia_generator[columns_to_clean] = eia_generator[columns_to_clean].replace(r'^\s*$', np.nan, regex=True)

# now define full dtype dictionary
eia_generator_type_dict = {
    "Year": "Int64", 
    "Utility ID": "string",
    "Utility Name": "string",
    "Plant Code": "Int64",  
    "Plant Name": "string",
    "State": "string",
    "Generator ID": "string",
    "Technology": "string",
    "Prime Mover": "string",
    "Nameplate Capacity (MW)": "float64",
    "Nameplate Power Factor": "float64", 
    "Summer Capacity (MW)": "float64",  
    "Winter Capacity (MW)": "float64",
    "Minimum Load (MW)": "float64",
    "Operating Year": "Int64",
    "Planned Retirement Year": "float64"
}

# filter needed cols
eia_generator = eia_generator[["Year","Utility ID","Utility Name", "Plant Code", "Plant Name", "State", "Generator ID", "Technology", "Prime Mover", "Nameplate Capacity (MW)", "Nameplate Power Factor", "Summer Capacity (MW)", "Winter Capacity (MW)", "Minimum Load (MW)", "Operating Year", "Planned Retirement Year"]]

# Artificially extend the unit_level dataframe to include 2024 data (just copy the 2023 data and set Year to 2024)
eia_generator_2023 = eia_generator[eia_generator["Year"] == 2023].copy()
eia_generator_2024 = eia_generator_2023.copy()
eia_generator_2024["Year"] = 2024
eia_generator = pd.concat([eia_generator, eia_generator_2024], ignore_index=True)

# cast all dtypes according to the dictionary
eia_generator = eia_generator.astype(eia_generator_type_dict)
eia_generator


,Year,Utility ID,Utility Name,Plant Code,Plant Name,State,Generator ID,Technology,Prime Mover,Nameplate Capacity (MW),Nameplate Power Factor,Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Operating Year,Planned Retirement Year
0,2019,12199,Montana-Dakota Utilities Co,2790,R M Heskett,ND,1,Conventional Steam Coal,ST,40.0,1.00,29.5,29.5,6.5,1954,2022.0
1,2019,12199,Montana-Dakota Utilities Co,2790,R M Heskett,ND,2,Conventional Steam Coal,ST,75.0,0.85,74.8,74.8,29.3,1963,2022.0
2,2019,12199,Montana-Dakota Utilities Co,2790,R M Heskett,ND,3,Natural Gas Fired Combustion Turbine,GT,88.0,0.85,88.0,101.0,8.0,2014,NaN
3,2019,1307,Basin Electric Power Coop,2817,Leland Olds,ND,1,Conventional Steam Coal,ST,216.0,0.85,221.0,221.0,110.0,1966,NaN
4,2019,1307,Basin Electric Power Coop,2817,Leland Olds,ND,2,Conventional Steam Coal,ST,440.0,0.90,445.0,445.0,220.0,1975,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
203,2024,1307,Basin Electric Power Coop,57943,Lonesome Creek Station,ND,2,Natural Gas Fired Combustion Turbine,GT,60.5,0.85,40.0,45.0,30.0,2015,NaN
204,2024,1307,Basin Electric Power Coop,57943,Lonesome Creek Station,ND,3,Natural Gas Fired Combustion Turbine,GT,60.5,0.85,40.0,45.0,30.0,2015,NaN
205,2024,1307,Basin Electric Power Coop,57943,Lonesome Creek Station,ND,4,Natural Gas Fired Combustion Turbine,GT,60.5,0.85,40.0,45.0,30.0,2017,NaN
206,2024,1307,Basin Electric Power Coop,57943,Lonesome Creek Station,ND,5,Natural Gas Fired Combustion Turbine,GT,60.5,0.85,40.0,45.0,30.0,2017,NaN


In [133]:
crosswalk = pd.read_csv("epa_eia_crosswalk.csv")
crosswalk = crosswalk[(crosswalk["CAMD_STATE"] == "NY") | (crosswalk["CAMD_STATE"] == "ND") | (crosswalk["CAMD_STATE"] == "WY")] # filter for the states we need
crosswalk[crosswalk["CAMD_PLANT_ID"] == 2490]

# the col "CAMD_STATUS_DATE" gives the date where changes where last made for this plant id and unit id combo. we have to create new rows for every individual year up until the next change and so on with an additional col "Year"
# to match with the campd data
# join on CAMP_PLANT_ID, CAMD_UNIT_ID and add CAMD_NAMEPLATE_CAPACITY, CAMD_STATUS_DATE, EIA_NAMEPLATE_CAPACITY, EIA_BOILER_ID

,SEQUENCE_NUMBER,CAMD_STATE,CAMD_FACILITY_NAME,CAMD_PLANT_ID,CAMD_UNIT_ID,CAMD_GENERATOR_ID,CAMD_NAMEPLATE_CAPACITY,CAMD_FUEL_TYPE,CAMD_LATITUDE,CAMD_LONGITUDE,...,EIA_LATITUDE,EIA_LONGITUDE,EIA_RETIRE_YEAR,PLANT_ID_CHANGE_FLAG,MOD_EIA_PLANT_ID,MOD_EIA_BOILER_ID,MOD_EIA_GENERATOR_ID_BOILER,MOD_EIA_GENERATOR_ID_GEN,MATCH_TYPE_GEN,MATCH_TYPE_BOILER
1772,1773,NY,Arthur Kill,2490,20,2,376.2,Pipeline Natural Gas,40.5915,-74.2027,...,40.591564,-74.200035,0.0,0.0,2490.0,20,2,2,3_1_Generator (generators) match on plant and ...,6_1_EnviroAssoc (boilers and generators) match...
1773,1774,NY,Arthur Kill,2490,30,3,535.5,Pipeline Natural Gas,40.5915,-74.2027,...,40.591564,-74.200035,0.0,0.0,2490.0,30,3,3,3_1_Generator (generators) match on plant and ...,6_1_EnviroAssoc (boilers and generators) match...
1774,1775,NY,Arthur Kill,2490,CT0001,GT1,16.3,Pipeline Natural Gas,40.5915,-74.2027,...,40.591564,-74.200035,0.0,0.0,2490.0,NaN,NaN,GT1,3_1_Generator (generators) match on plant and ...,NaN


In [134]:
# create yearly rows manually

crosswalk["CAMD_STATUS_DATE"] = pd.to_datetime(crosswalk["CAMD_STATUS_DATE"])

crosswalk_sorted = crosswalk.sort_values(by=["CAMD_PLANT_ID", "CAMD_UNIT_ID", "CAMD_STATUS_DATE"])

# Define end year up to which we generate rows
default_end_year = 2024

expanded_rows = [] # save rows for expansion

# Group by plant/unit and generate yearly rows
for (plant_id, unit_id), group in crosswalk_sorted.groupby(["CAMD_PLANT_ID", "CAMD_UNIT_ID"]):
    group = group.sort_values("CAMD_STATUS_DATE").reset_index(drop=True)

    for i in range(len(group)):
        start_year = group.loc[i, "CAMD_STATUS_DATE"].year
        end_year = (group.loc[i + 1, "CAMD_STATUS_DATE"].year 
                    if i + 1 < len(group) 
                    else default_end_year + 1)  # +1 so we include the final year

        for year in range(start_year, end_year):
            row = group.loc[i].copy()
            row["Year"] = year
            expanded_rows.append(row)

# Combine into new df
expanded_crosswalk = pd.DataFrame(expanded_rows)

expanded_crosswalk = expanded_crosswalk[expanded_crosswalk["Year"] >= 2019]
pd.set_option('display.max_columns', None)
expanded_crosswalk.sort_values(by=["CAMD_PLANT_ID", "CAMD_UNIT_ID", "Year"])

,SEQUENCE_NUMBER,CAMD_STATE,CAMD_FACILITY_NAME,CAMD_PLANT_ID,CAMD_UNIT_ID,CAMD_GENERATOR_ID,CAMD_NAMEPLATE_CAPACITY,CAMD_FUEL_TYPE,CAMD_LATITUDE,CAMD_LONGITUDE,CAMD_STATUS,CAMD_STATUS_DATE,CAMD_RETIRE_YEAR,MOD_CAMD_UNIT_ID,MOD_CAMD_GENERATOR_ID,EIA_STATE,EIA_PLANT_NAME,EIA_PLANT_ID,EIA_GENERATOR_ID,EIA_NAMEPLATE_CAPACITY,EIA_BOILER_ID,EIA_UNIT_TYPE,EIA_FUEL_TYPE,EIA_LATITUDE,EIA_LONGITUDE,EIA_RETIRE_YEAR,PLANT_ID_CHANGE_FLAG,MOD_EIA_PLANT_ID,MOD_EIA_BOILER_ID,MOD_EIA_GENERATOR_ID_BOILER,MOD_EIA_GENERATOR_ID_GEN,MATCH_TYPE_GEN,MATCH_TYPE_BOILER,Year
0,1769,NY,Danskammer Generating Station,2480,1,1,72.0,Pipeline Natural Gas,41.573,-73.9646,OPR,2014-09-03,0,1,1,NY,Danskammer Generating Station,2480.0,1,72.0,1,ST,NG,41.571247,-73.974981,0.0,0.0,2480.0,1,1,1,3_1_Generator (generators) match on plant and ...,6_1_EnviroAssoc (boilers and generators) match...,2019
0,1769,NY,Danskammer Generating Station,2480,1,1,72.0,Pipeline Natural Gas,41.573,-73.9646,OPR,2014-09-03,0,1,1,NY,Danskammer Generating Station,2480.0,1,72.0,1,ST,NG,41.571247,-73.974981,0.0,0.0,2480.0,1,1,1,3_1_Generator (generators) match on plant and ...,6_1_EnviroAssoc (boilers and generators) match...,2020
0,1769,NY,Danskammer Generating Station,2480,1,1,72.0,Pipeline Natural Gas,41.573,-73.9646,OPR,2014-09-03,0,1,1,NY,Danskammer Generating Station,2480.0,1,72.0,1,ST,NG,41.571247,-73.974981,0.0,0.0,2480.0,1,1,1,3_1_Generator (generators) match on plant and ...,6_1_EnviroAssoc (boilers and generators) match...,2021
0,1769,NY,Danskammer Generating Station,2480,1,1,72.0,Pipeline Natural Gas,41.573,-73.9646,OPR,2014-09-03,0,1,1,NY,Danskammer Generating Station,2480.0,1,72.0,1,ST,NG,41.571247,-73.974981,0.0,0.0,2480.0,1,1,1,3_1_Generator (generators) match on plant and ...,6_1_EnviroAssoc (boilers and generators) match...,2022
0,1769,NY,Danskammer Generating Station,2480,1,1,72.0,Pipeline Natural Gas,41.573,-73.9646,OPR,2014-09-03,0,1,1,NY,Danskammer Generating Station,2480.0,1,72.0,1,ST,NG,41.571247,-73.974981,0.0,0.0,2480.0,1,1,1,3_1_Generator (generators) match on plant and ...,6_1_EnviroAssoc (boilers and generators) match...,2023
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,6926,NY,Ravenswood Steam Plant,880100,BLR004,NaN,NaN,Residual Oil,40.760,-73.7500,OPR,1947-12-01,0,BLR004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Manual CAMD Excluded,Manual CAMD Excluded,2020
0,6926,NY,Ravenswood Steam Plant,880100,BLR004,NaN,NaN,Residual Oil,40.760,-73.7500,OPR,1947-12-01,0,BLR004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Manual CAMD Excluded,Manual CAMD Excluded,2021
0,6926,NY,Ravenswood Steam Plant,880100,BLR004,NaN,NaN,Residual Oil,40.760,-73.7500,OPR,1947-12-01,0,BLR004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Manual CAMD Excluded,Manual CAMD Excluded,2022
0,6926,NY,Ravenswood Steam Plant,880100,BLR004,NaN,NaN,Residual Oil,40.760,-73.7500,OPR,1947-12-01,0,BLR004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Manual CAMD Excluded,Manual CAMD Excluded,2023


In [135]:
expanded_crosswalk["EIA_PLANT_ID"] = expanded_crosswalk["EIA_PLANT_ID"].astype("Int64")
expanded_crosswalk["EIA_GENERATOR_ID"] = expanded_crosswalk["EIA_GENERATOR_ID"].astype("string")
expanded_crosswalk["Year"] = expanded_crosswalk["Year"].astype("Int64")

In [136]:
# join crosswalk with additional eia_generator data
combined_crosswalk = expanded_crosswalk.merge(eia_generator, how="left", left_on=["EIA_PLANT_ID", "EIA_GENERATOR_ID", "Year"], right_on=["Plant Code", "Generator ID", "Year"])
combined_crosswalk

,SEQUENCE_NUMBER,CAMD_STATE,CAMD_FACILITY_NAME,CAMD_PLANT_ID,CAMD_UNIT_ID,CAMD_GENERATOR_ID,CAMD_NAMEPLATE_CAPACITY,CAMD_FUEL_TYPE,CAMD_LATITUDE,CAMD_LONGITUDE,CAMD_STATUS,CAMD_STATUS_DATE,CAMD_RETIRE_YEAR,MOD_CAMD_UNIT_ID,MOD_CAMD_GENERATOR_ID,EIA_STATE,EIA_PLANT_NAME,EIA_PLANT_ID,EIA_GENERATOR_ID,EIA_NAMEPLATE_CAPACITY,EIA_BOILER_ID,EIA_UNIT_TYPE,EIA_FUEL_TYPE,EIA_LATITUDE,EIA_LONGITUDE,EIA_RETIRE_YEAR,PLANT_ID_CHANGE_FLAG,MOD_EIA_PLANT_ID,MOD_EIA_BOILER_ID,MOD_EIA_GENERATOR_ID_BOILER,MOD_EIA_GENERATOR_ID_GEN,MATCH_TYPE_GEN,MATCH_TYPE_BOILER,Year,Utility ID,Utility Name,Plant Code,Plant Name,State,Generator ID,Technology,Prime Mover,Nameplate Capacity (MW),Nameplate Power Factor,Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Operating Year,Planned Retirement Year
0,1769,NY,Danskammer Generating Station,2480,1,1,72.0,Pipeline Natural Gas,41.573,-73.9646,OPR,2014-09-03,0,1,1,NY,Danskammer Generating Station,2480,1,72.0,1,ST,NG,41.571247,-73.974981,0.0,0.0,2480.0,1,1,1,3_1_Generator (generators) match on plant and ...,6_1_EnviroAssoc (boilers and generators) match...,2019,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
1,1769,NY,Danskammer Generating Station,2480,1,1,72.0,Pipeline Natural Gas,41.573,-73.9646,OPR,2014-09-03,0,1,1,NY,Danskammer Generating Station,2480,1,72.0,1,ST,NG,41.571247,-73.974981,0.0,0.0,2480.0,1,1,1,3_1_Generator (generators) match on plant and ...,6_1_EnviroAssoc (boilers and generators) match...,2020,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
2,1769,NY,Danskammer Generating Station,2480,1,1,72.0,Pipeline Natural Gas,41.573,-73.9646,OPR,2014-09-03,0,1,1,NY,Danskammer Generating Station,2480,1,72.0,1,ST,NG,41.571247,-73.974981,0.0,0.0,2480.0,1,1,1,3_1_Generator (generators) match on plant and ...,6_1_EnviroAssoc (boilers and generators) match...,2021,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
3,1769,NY,Danskammer Generating Station,2480,1,1,72.0,Pipeline Natural Gas,41.573,-73.9646,OPR,2014-09-03,0,1,1,NY,Danskammer Generating Station,2480,1,72.0,1,ST,NG,41.571247,-73.974981,0.0,0.0,2480.0,1,1,1,3_1_Generator (generators) match on plant and ...,6_1_EnviroAssoc (boilers and generators) match...,2022,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
4,1769,NY,Danskammer Generating Station,2480,1,1,72.0,Pipeline Natural Gas,41.573,-73.9646,OPR,2014-09-03,0,1,1,NY,Danskammer Generating Station,2480,1,72.0,1,ST,NG,41.571247,-73.974981,0.0,0.0,2480.0,1,1,1,3_1_Generator (generators) match on plant and ...,6_1_EnviroAssoc (boilers and generators) match...,2023,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2315,6926,NY,Ravenswood Steam Plant,880100,BLR004,NaN,NaN,Residual Oil,40.760,-73.7500,OPR,1947-12-01,0,BLR004,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Manual CAMD Excluded,Manual CAMD Excluded,2020,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
2316,6926,NY,Ravenswood Steam Plant,880100,BLR004,NaN,NaN,Residual Oil,40.760,-73.7500,OPR,1947-12-01,0,BLR004,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Manual CAMD Excluded,Manual CAMD Excluded,2021,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
2317,6926,NY,Ravenswood Steam Plant,880100,BLR004,NaN,NaN,Residual Oil,40.760,-73.7500,OPR,1947-12-01,0,BLR004,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Manual CAMD Excluded,Manual CAMD Excluded,2022,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
2318,6926,NY,Ravenswood Steam Plant,880100,BLR004,NaN,NaN,Residual Oil,40.760,-73.7500,OPR,1947-12-01,0,BLR004,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Manual CAMD Excluded,Manual CAMD E

In [138]:
# read in cleaned dfs
campd_nd = pd.read_csv("ND/Hourly/Concatted File/nd_2019_2024_complete.csv", dtype=campd_type_dict)

In [139]:
combined_crosswalk = expanded_crosswalk.merge(eia_generator, how="right", left_on=["EIA_PLANT_ID", "EIA_GENERATOR_ID", "Year"], right_on=["Plant Code", "Generator ID", "Year"])
combined_crosswalk

,SEQUENCE_NUMBER,CAMD_STATE,CAMD_FACILITY_NAME,CAMD_PLANT_ID,CAMD_UNIT_ID,CAMD_GENERATOR_ID,CAMD_NAMEPLATE_CAPACITY,CAMD_FUEL_TYPE,CAMD_LATITUDE,CAMD_LONGITUDE,CAMD_STATUS,CAMD_STATUS_DATE,CAMD_RETIRE_YEAR,MOD_CAMD_UNIT_ID,MOD_CAMD_GENERATOR_ID,EIA_STATE,EIA_PLANT_NAME,EIA_PLANT_ID,EIA_GENERATOR_ID,EIA_NAMEPLATE_CAPACITY,EIA_BOILER_ID,EIA_UNIT_TYPE,EIA_FUEL_TYPE,EIA_LATITUDE,EIA_LONGITUDE,EIA_RETIRE_YEAR,PLANT_ID_CHANGE_FLAG,MOD_EIA_PLANT_ID,MOD_EIA_BOILER_ID,MOD_EIA_GENERATOR_ID_BOILER,MOD_EIA_GENERATOR_ID_GEN,MATCH_TYPE_GEN,MATCH_TYPE_BOILER,Year,Utility ID,Utility Name,Plant Code,Plant Name,State,Generator ID,Technology,Prime Mover,Nameplate Capacity (MW),Nameplate Power Factor,Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Operating Year,Planned Retirement Year
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2019,12199,Montana-Dakota Utilities Co,2790,R M Heskett,ND,1,Conventional Steam Coal,ST,40.0,1.00,29.5,29.5,6.5,1954,2022.0
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2019,12199,Montana-Dakota Utilities Co,2790,R M Heskett,ND,2,Conventional Steam Coal,ST,75.0,0.85,74.8,74.8,29.3,1963,2022.0
2,2050.0,ND,R M Heskett,2790.0,CT6,3,88.0,Pipeline Natural Gas,46.8669,-100.8839,OPR,2014-06-10,0.0,CT6,3,ND,R M Heskett,2790,3,88.0,NaN,GT,NG,46.866900,-100.883600,0.0,0.0,2790.0,NaN,NaN,3,3_1_Generator (generators) match on plant and ...,NaN,2019,12199,Montana-Dakota Utilities Co,2790,R M Heskett,ND,3,Natural Gas Fired Combustion Turbine,GT,88.0,0.85,88.0,101.0,8.0,2014,NaN
3,2051.0,ND,Leland Olds,2817.0,1,1,216.0,Coal,47.2819,-101.3194,OPR,1966-01-30,0.0,1,1,ND,Leland Olds,2817,1,216.0,1,ST,RC,47.280769,-101.321213,0.0,0.0,2817.0,1,1,1,3_1_Generator (generators) match on plant and ...,6_1_EnviroAssoc (boilers and generators) match...,2019,1307,Basin Electric Power Coop,2817,Leland Olds,ND,1,Conventional Steam Coal,ST,216.0,0.85,221.0,221.0,110.0,1966,NaN
4,2052.0,ND,Leland Olds,2817.0,2,2,440.0,Coal,47.2819,-101.3194,OPR,1975-12-15,0.0,2,2,ND,Leland Olds,2817,2,440.0,2,ST,RC,47.280769,-101.321213,0.0,0.0,2817.0,2,2,2,3_1_Generator (generators) match on plant and ...,6_1_EnviroAssoc (boilers and generators) match...,2019,1307,Basin Electric Power Coop,2817,Leland Olds,ND,2,Conventional Steam Coal,ST,440.0,0.90,445.0,445.0,220.0,1975,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
203,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024,1307,Basin Electric Power Coop,57943,Lonesome Creek Station,ND,2,Natural Gas Fired Combustion Turbine,GT,60.5,0.85,40.0,45.0,30.0,2015,NaN
204,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024,1307,Basin Electric Power Coop,57943,Lonesome Creek Station,ND,3,Natural Gas Fired Combustion Turbine,GT,60.5,0.85,40.0,45.0,30.0,2015,NaN
205,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024,1307,Basin Electric Power Coop,57943,Lonesome Creek Station,ND,4,Natural Gas Fired Combustion Turbine,GT,60.5,0.85,40.0,45.0,30.0,2017,NaN
206,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024,1307,Basin Electric Power Coop,57943,Lonesome Creek Station,ND,5,Natural Gas Fired Combustion Turbine,GT,60.5,0.85,40.0,45.0,30.0,2017,NaN


In [140]:
combined_crosswalk = combined_crosswalk[['CAMD_PLANT_ID',
       'CAMD_UNIT_ID', 'CAMD_GENERATOR_ID', 'CAMD_NAMEPLATE_CAPACITY', 'CAMD_STATUS_DATE', 'CAMD_RETIRE_YEAR', 'EIA_PLANT_ID',
       'EIA_GENERATOR_ID', 'EIA_NAMEPLATE_CAPACITY', 'EIA_BOILER_ID', 'EIA_RETIRE_YEAR', 'Year', 'Utility ID', 'Utility Name', 'Plant Code', 'Plant Name',
       'State', 'Generator ID', 'Technology', 'Prime Mover',
       'Nameplate Capacity (MW)', 'Nameplate Power Factor',
       'Summer Capacity (MW)', 'Winter Capacity (MW)', 'Minimum Load (MW)',
       'Operating Year', 'Planned Retirement Year']]

In [141]:
# bring to CAMD unit-level

agg_funcs = {
    'Nameplate Capacity (MW)': 'sum',
    'Summer Capacity (MW)': 'sum',
    'Winter Capacity (MW)': 'sum',
    'Minimum Load (MW)': 'sum',
    'Operating Year': ['min', 'max', 'mean'], # get operating year of the oldest (min) and newest (max) generator in the unit as well as the average across all gens
    'Planned Retirement Year': 'max',
}

unit_level = combined_crosswalk.groupby(['Year', 'EIA_PLANT_ID', 'CAMD_UNIT_ID']).agg(agg_funcs).reset_index()

display(unit_level)

# flatten multi-index columns (only keep the aggregation operator (min, max, mean) for Operating Year)
unit_level.columns = [
    f"{col[0]}_{col[1]}" if col[0] == "Operating Year" else col[0]
    for col in unit_level.columns.to_flat_index()
]

# set data types again for the final unit_level dataframe
unit_level = unit_level.astype({
    'Year': 'Int64',
    'EIA_PLANT_ID': 'Int64',
    'CAMD_UNIT_ID': 'string',
    'Nameplate Capacity (MW)': 'float64',
    'Summer Capacity (MW)': 'float64',
    'Winter Capacity (MW)': 'float64',
    'Minimum Load (MW)': 'float64',
    'Operating Year_min': 'Int64',
    'Operating Year_max': 'Int64',
    'Operating Year_mean': 'Int64',
    'Planned Retirement Year': 'Int64'
})

unit_level

Year EIA_PLANT_ID CAMD_UNIT_ID Nameplate Capacity (MW)  \
                                                       sum   
0   2019         2790          CT6                    88.0   
1   2019         2817            1                   216.0   
2   2019         2817            2                   440.0   
3   2019         2823           B1                   257.0   
4   2019         2823           B2                   477.0   
..   ...          ...          ...                     ...   
61  2024         6030            2                   604.8   
62  2024         6469           B1                   477.0   
63  2024         6469           B2                   477.0   
64  2024         8222           B1                   450.0   
65  2024        56786          001                   106.2   

   Summer Capacity (MW) Winter Capacity (MW) Minimum Load (MW) Operating Year  \
                    sum                  sum               sum            min   
0                  88.0                101.0               8.0           2014   
1                 221.0                221.0             110.0           1966   
2                 445.0                445.0             220.0           1975   
3                 237.0                237.0             150.0           1970   
4                 447.0                455.0             270.0           1977   
..                  ...                  ...               ...            ...   
61                573.9                573.9             285.0           1980   
62                450.0                450.0             180.0           1984   
63                450.0                450.0             180.0           1986   
64                427.0                427.0             150.0           1981   
65                 97.1                 97.1              25.0           2014   

                 Planned Retirement Year  
     max    mean                     max  
0   2014  2014.0                     NaN  
1   1966  1966.0                     NaN  
2   1975  1975.0                     NaN  
3   1970  1970.0                     NaN  
4   1977  1977.0                     NaN  
..   ...     ...                     ...  
61  1980  1980.0                     NaN  
62  1984  1984.0                     NaN  
63  1986  1986.0                     NaN  
64  1981  1981.0                     NaN  
65  2014  2014.0                     NaN  

[66 rows x 11 columns]

,Year,EIA_PLANT_ID,CAMD_UNIT_ID,Nameplate Capacity (MW),Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Operating Year_min,Operating Year_max,Operating Year_mean,Planned Retirement Year
0,2019,2790,CT6,88.0,88.0,101.0,8.0,2014,2014,2014,<NA>
1,2019,2817,1,216.0,221.0,221.0,110.0,1966,1966,1966,<NA>
2,2019,2817,2,440.0,445.0,445.0,220.0,1975,1975,1975,<NA>
3,2019,2823,B1,257.0,237.0,237.0,150.0,1970,1970,1970,<NA>
4,2019,2823,B2,477.0,447.0,455.0,270.0,1977,1977,1977,<NA>
...,...,...,...,...,...,...,...,...,...,...,...
61,2024,6030,2,604.8,573.9,573.9,285.0,1980,1980,1980,<NA>
62,2024,6469,B1,477.0,450.0,450.0,180.0,1984,1984,1984,<NA>
63,2024,6469,B2,477.0,450.0,450.0,180.0,1986,1986,1986,<NA>
64,2024,8222,B1,450.0,427.0,427.0,150.0,1981,1981,1981,<NA>


In [142]:
# Calc min, max, and mean ages of the generators in the unit
unit_level['unit_age_min'] = unit_level["Year"] - unit_level['Operating Year_min']
unit_level['unit_age_max'] = unit_level["Year"] - unit_level['Operating Year_max']
unit_level['unit_age_mean'] = unit_level["Year"] - unit_level['Operating Year_mean']

In [143]:
unit_level = unit_level.rename(columns={
    "EIA_PLANT_ID": "Facility ID",
    "CAMD_UNIT_ID": "Unit ID"
})

In [144]:
# Merge each with unit_level
complete_campd_nd = pd.merge(campd_nd, unit_level, on=['Year', 'Facility ID', 'Unit ID'], how='left')

In [145]:
# Calculate differences between aggregated EIA Nameplate Capacity and CAMD Nameplate Capacity

# Create a column for absolute difference
complete_campd_nd["Capacity Difference"] = (
    complete_campd_nd["Nameplate Capacity (MWe)"] - complete_campd_nd["Nameplate Capacity (MW)"]
).abs()

# Drop rows with NaNs in either of the capacity columns
diff_df = complete_campd_nd.dropna(subset=["Nameplate Capacity (MWe)", "Nameplate Capacity (MW)"])

# Calculate average difference
average_difference_nd = diff_df["Capacity Difference"].mean()
print(f"Average Capacity Difference ND: {average_difference_nd:.2f} MW")

Average Capacity Difference ND: 13.84 MW


In [147]:
# fill EIA columns with na if it capacity difference exceeds a certain threshold

# rename the EIA column to make it clearer
for df in [complete_campd_nd]:
    df.rename(columns={
        "Nameplate Capacity (MW)": "Nameplate Capacity (MW) - EIA"
    }, inplace=True)

threshold = 30 
columns_to_null = [
    "Nameplate Capacity (MW) - EIA", 
    "Summer Capacity (MW)_y",
    "Winter Capacity (MW)_y",
    "Minimum Load (MW)_y"
]

# Recalculate capacity difference
complete_campd_nd["Capacity Difference"] = (
    complete_campd_nd["Nameplate Capacity (MWe)"] - complete_campd_nd["Nameplate Capacity (MW) - EIA"]
).abs()

# Identify rows with unreasonable differences
condition = complete_campd_nd["Capacity Difference"] > threshold

# Set selected columns to NaN where condition is met
complete_campd_nd.loc[condition, columns_to_null] = np.nan

In [ ]:
# Calculate capacity difference again for comparison

# Create a column for absolute difference
complete_campd_nd["Capacity Difference"] = (
    complete_campd_nd["Nameplate Capacity (MWe)"] - complete_campd_nd["Nameplate Capacity (MW) - EIA"]
).abs()

# Drop rows with NaNs in either of the capacity columns
diff_df = complete_campd_nd.dropna(subset=["Nameplate Capacity (MWe)", "Nameplate Capacity (MW) - EIA"])

# Calculate average difference
average_difference_nd = diff_df["Capacity Difference"].mean()
print(f"Average Capacity Difference ND: {average_difference_nd:.2f} MW")

Average Capacity Difference ND: 1.91 MW


In [149]:
# Rename crosswalk cols and remove manual EIA transformation cols

# Define the renaming mapping
rename_cols = {
    'Summer Capacity (MW)_y': 'Summer Capacity (MW)',
    'Winter Capacity (MW)_y': 'Winter Capacity (MW)',
    'Minimum Load (MW)_y': 'Minimum Load (MW)',
    'Operating Year_min': 'Operating Year_min',
    'Operating Year_max': 'Operating Year_max',
    'Operating Year_mean': 'Operating Year_mean',
    'Planned Retirement Year_y': 'Planned Retirement Year',
    'unit_age_min_y': 'unit_age_min',
    'unit_age_max_y': 'unit_age_max',
    'unit_age_mean_y': 'unit_age_mean',
}


In [150]:
# Clean ND dataset
complete_campd_nd = complete_campd_nd.drop(columns=[
    'Summer Capacity (MW)_x', 'Winter Capacity (MW)_x',
    'Minimum Load (MW)_x', 'Planned Retirement Year_x',
    'unit_age_min_x', 'unit_age_max_x', 'unit_age_mean_x',
    'Capacity Difference', 'Nameplate Capacity (MW) - EIA'
])
complete_campd_nd = complete_campd_nd.rename(columns=rename_cols)

In [151]:
complete_campd_nd.to_csv('ND/Hourly/Concatted File/nd_2019_2024_complete_cleaned.csv', index=False)

In [152]:
complete_campd_nd

,State,Facility Name,Facility ID,Unit ID,Date,Hour,Operating Time,Gross Load (MW),Steam Load (1000 lb/hr),SO2 Mass (lbs),SO2 Mass Measure Indicator,SO2 Rate (lbs/mmBtu),SO2 Rate Measure Indicator,CO2 Mass (short tons),CO2 Mass Measure Indicator,CO2 Rate (short tons/mmBtu),CO2 Rate Measure Indicator,NOx Mass (lbs),NOx Mass Measure Indicator,NOx Rate (lbs/mmBtu),NOx Rate Measure Indicator,Heat Input (mmBtu),Heat Input Measure Indicator,Primary Fuel Type,Secondary Fuel Type,Unit Type,SO2 Controls,NOx Controls,Program Code,Year,Source Category,Latitude,Longitude,SO2 Phase,NOx Phase,Operating Status,Max Hourly HI Rate (mmBtu/hr),Associated Generators & Nameplate Capacity (MWe),Nameplate Capacity (MWe),Month,Day,hourly_capacity_factor,hourly_ramping_factor,Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Operating Year_min,Operating Year_max,Operating Year_mean,Planned Retirement Year,unit_age_min,unit_age_max,unit_age_mean
0,ND,R M Heskett,2790,B2,2019-01-01,0,1.0,59.0,NaN,251.7,Measured,0.395,Calculated,69.4,Measured,0.109,Calculated,207.058,Calculated,0.325,Measured,637.1,Measured,Coal,Nothing,Bubbling fluidized bed boiler,Fluidized Bed Limestone Injection,Nothing,"ARP, MATS",2019,Electric Utility,46.8669,-100.8839,Phase 2,Nothing,Operating,1257.0,2 (75),75.0,1,1,0.786667,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,ND,R M Heskett,2790,B2,2019-01-01,1,1.0,59.0,NaN,264.7,Measured,0.411,Calculated,70.0,Measured,0.109,Calculated,200.066,Calculated,0.311,Measured,643.3,Measured,Coal,Nothing,Bubbling fluidized bed boiler,Fluidized Bed Limestone Injection,Nothing,"ARP, MATS",2019,Electric Utility,46.8669,-100.8839,Phase 2,Nothing,Operating,1257.0,2 (75),75.0,1,1,0.786667,0.000000,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,ND,R M Heskett,2790,B2,2019-01-01,2,1.0,60.0,NaN,302.6,Measured,0.464,Calculated,71.0,Measured,0.109,Calculated,194.415,Calculated,0.298,Measured,652.4,Measured,Coal,Nothing,Bubbling fluidized bed boiler,Fluidized Bed Limestone Injection,Nothing,"ARP, MATS",2019,Electric Utility,46.8669,-100.8839,Phase 2,Nothing,Operating,1257.0,2 (75),75.0,1,1,0.800000,0.013333,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,ND,R M Heskett,2790,B2,2019-01-01,3,1.0,59.0,NaN,229.3,Measured,0.360,Calculated,69.3,Measured,0.109,Calculated,206.895,Calculated,0.325,Measured,636.6,Measured,Coal,Nothing,Bubbling fluidized bed boiler,Fluidized Bed Limestone Injection,Nothing,"ARP, MATS",2019,Electric Utility,46.8669,-100.8839,Phase 2,Nothing,Operating,1257.0,2 (75),75.0,1,1,0.786667,-0.013333,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,ND,R M Heskett,2790,B2,2019-01-01,4,1.0,58.0,NaN,242.9,Measured,0.382,Calculated,69.2,Measured,0.109,Calculated,209.814,Calculated,0.330,Measured,635.8,Measured,Coal,Nothing,Bubbling fluidized bed boiler,Fluidized Bed Limestone Injection,Nothing,"ARP, MATS",2019,Electric Utility,46.8669,-100.8839,Phase 2,Nothing,Operating,1257.0,2 (75),75.0,1,1,0.773333,-0.013333,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1063123,ND,Lonesome Creek Station,57943,CT6,2024-12-31,19,0.0,0.0,NaN,NaN,<NA>,NaN,Missing,NaN,<NA>,NaN,Missing,NaN,<NA>,NaN,Missing,NaN,<NA>,Pipeline Natural Gas,Nothing,Combustion turbine,Nothing,Dry Low NOx Burners|Selective Catalytic Reduction,ARP,2024,Electric Utility,47.7966,-103.5783,Phase 2,Nothing,Operating,495.0,06 (45.2),45.2,12,31,NaN,0.000000,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1063124,ND,Lonesome Creek Station,57943,CT6,2024-12-31,20,0.0,0.0,NaN,NaN,<NA>,NaN,Missing,NaN,<NA>,NaN,Missing,NaN,<NA>,NaN,Missing,NaN,<NA>,Pipeline Natural Gas,Nothing,Combustion turbine,Nothing,Dry Low NOx Burners|Selective Catalytic Reduction,ARP,2024,Electric Utility,47.7966,-103.5783,Phase 2,Nothing,Operating,495.0,06 (45.2),45.2,12,31,NaN,0.000000,NaN,NaN,NaN